In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchsummary import summary

from ptflops import get_model_complexity_info

In [11]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [12]:
# **Классификация изображений с помощью сверточных нейронных сетей**

# В данном задании Вам необходимо разработать архитектуру сверточной ИНС, обеспечивающую наибольшую точность при ограничении на количество операций (FLOPs <= 0.707e6).
# Заготовка кода для выполнения задания приведена выше. Вашей задачей будет заполнить пропущеные места, которые отмечены ключевым словом *None*.
# Необходимая точность (accuracy) сети на датасете CIFAR100 - 30%
# Желаемая точность (accuracy) сети на датасете CIFAR100 - 45%

In [13]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") 
#надо переписать строчку, чтобы заработало с cuda

In [14]:
# Глобальные константы датасета
CLASSES = 10
BATCH_SIZE = 128
VAL_RATIO = 0.2 

In [15]:
# Загрузка данных MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Нормализация для MNIST
])

full_train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

# Тестовый набор (10k изображений) - используем стандартный тестовый набор MNIST
test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)


train_size = int((1 - VAL_RATIO) * len(full_train_dataset))
val_size = int(VAL_RATIO * len(full_train_dataset))

# Разделяем данные с фиксированным генератором
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset, 
    [train_size, val_size],
    generator=generator
)

# Создаем DataLoader'ы с фиксированным seed для shuffle
def seed_worker(worker_id):
    worker_seed = SEED
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4
)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Standard test samples: {len(test_dataset)}")

Train samples: 48000
Validation samples: 12000
Standard test samples: 10000


In [19]:
# Определение MLP модели
class MNISTMLP(nn.Module):
    def __init__(self):
        super(MNISTMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 512)  # MNIST изображения 28x28 пикселей
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, CLASSES)
        
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

model = MNISTMLP()

In [20]:
# Вычисление FLOPs
input_shape = (1, 28, 28)  # MNIST: 1 канал, 28x28 изображения
flops, params = get_model_complexity_info(model, input_shape, as_strings=False, print_per_layer_stat=False)
print(f"FLOPs: {(flops / 1e6):.4f}e6\n\n")

# Вывод информации о модели
summary(model, input_size=input_shape)

FLOPs: 0.5366e6


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
            Linear-3                  [-1, 256]         131,328
            Linear-4                   [-1, 10]           2,570
Total params: 535,818
Trainable params: 535,818
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 2.04
Estimated Total Size (MB): 2.06
----------------------------------------------------------------


In [ ]:
LEARNING_RATE = 1e-2

# Оптимизатор и функция потерь
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)
criterion = nn.CrossEntropyLoss()

In [ ]:
model.to(device)

def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        scheduler.step()


def evaluate(model, device, loader, criterion):
    model.eval()
    loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    loss /= len(loader.dataset)
    accuracy = 100. * correct / len(loader.dataset)
    return loss, accuracy

In [ ]:
best_accuracy = 0
for epoch in range(1, 21):
    train(model, device, train_loader, optimizer, criterion, epoch)
    val_loss, val_accuracy = evaluate(model, device, val_loader, criterion)
    
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
    
    print(f'Epoch {epoch}: Validation Accuracy: {val_accuracy:.2f}%')

Epoch 1: Validation Accuracy: 92.49%
Epoch 2: Validation Accuracy: 93.93%
Epoch 3: Validation Accuracy: 95.37%
Epoch 4: Validation Accuracy: 95.94%
Epoch 5: Validation Accuracy: 95.95%
Epoch 6: Validation Accuracy: 96.61%
Epoch 7: Validation Accuracy: 96.88%
Epoch 8: Validation Accuracy: 96.72%
Epoch 9: Validation Accuracy: 96.84%
Epoch 10: Validation Accuracy: 96.95%
Epoch 11: Validation Accuracy: 97.14%
Epoch 12: Validation Accuracy: 97.00%
Epoch 13: Validation Accuracy: 97.19%
Epoch 14: Validation Accuracy: 97.19%
Epoch 15: Validation Accuracy: 97.20%
Epoch 16: Validation Accuracy: 97.17%
Epoch 17: Validation Accuracy: 97.19%
Epoch 18: Validation Accuracy: 97.20%
Epoch 19: Validation Accuracy: 97.19%
Epoch 20: Validation Accuracy: 97.22%


In [ ]:
test_loss, test_accuracy = evaluate(model, device, test_loader, criterion)
print(f'Standard Test Accuracy: {test_accuracy:.2f}%')

Standard Test Accuracy: 97.25%
